In [0]:
import sys, os
sys.path.append(os.path.abspath("../common"))
from config import (
    CATALOG, RAW_SCHEMA, BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA,
    CONTROL_SCHEMA, RAW_VOLUME, WATERMARK_TABLE, RUN_LOG_TABLE, FHIR_RESOURCES
)

In [0]:
# COMMAND ----------
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
for schema in [RAW_SCHEMA, BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA, CONTROL_SCHEMA]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")

In [0]:
# Volume for immutable raw JSON (as-is API responses)
spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {CATALOG}.{RAW_SCHEMA}.raw_data
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {WATERMARK_TABLE} (
    resource_type       STRING,
    last_run_started_at TIMESTAMP,
    last_run_finished_at TIMESTAMP,
    last_updated_watermark STRING,   -- max FHIR meta.lastUpdated ingested
    status               STRING,     -- SUCCESS / FAILED
    rows_ingested         BIGINT
) USING DELTA
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {RUN_LOG_TABLE} (
    run_id           STRING,
    resource_type    STRING,
    layer            STRING,        -- RAW / BRONZE / SILVER / GOLD
    api_url_or_params STRING,
    extraction_timestamp TIMESTAMP,
    page_number      INT,
    entry_count      INT,
    status           STRING,
    error_message    STRING
) USING DELTA
""")

In [0]:
for r in FHIR_RESOURCES:
    exists = spark.sql(
        f"SELECT 1 FROM {WATERMARK_TABLE} WHERE resource_type = '{r}'"
    ).count()
    if exists == 0:
        spark.sql(f"""
            INSERT INTO {WATERMARK_TABLE}
            VALUES ('{r}', NULL, NULL, NULL, 'NEVER_RUN', 0)
        """)

print("Setup complete: catalog, schemas, volume, and control tables are ready.")